# Análise Cyclistic — Q1 2019 & 2020

Projeto de conclusão do **Certificado Profissional de Análise de Dados do Google**.  
**Pergunta central:** Como membros anuais e usuários casuais utilizam as bicicletas Cyclistic de forma diferente?

---
## 1. Importando Bibliotecas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')

## 2. Configuração dos Caminhos

Atualize as variáveis abaixo com o caminho local dos arquivos CSV.  
Veja `dados/fonte_dos_dados.md` para instruções de download.

In [ ]:
CAMINHO_2019 = r"caminho/para/Divvy_Trips_2019_Q1.csv"
CAMINHO_2020 = r"caminho/para/Divvy_Trips_2020_Q1.csv"

df_2019 = pd.read_csv(CAMINHO_2019)
df_2020 = pd.read_csv(CAMINHO_2020)

## 3. Verificando Compatibilidade dos Datasets

In [ ]:
print("Colunas 2019:", list(df_2019.columns))
print("Colunas 2020:", list(df_2020.columns))

Observa-se que a arquitetura do banco de dados da Cyclistic foi **alterada** entre 2019 e 2020.

In [ ]:
print(df_2019.info())
print(df_2020.info())

Os tipos de dados também mudaram — em destaque, o ID passou a ser do tipo `object` em 2020.

In [ ]:
display(df_2019.head())
display(df_2020.head())

## 4. Limpeza e Padronização

### 4.1 Removendo colunas incompatíveis

- **2019:** remove colunas demográficas (`gender`, `birthyear`) e de bicicleta (`bikeid`, `tripduration`) ausentes em 2020
- **2020:** remove colunas de geolocalização e tipo de bicicleta ausentes em 2019

In [ ]:
df_2019 = df_2019.drop(columns=['gender', 'birthyear', 'bikeid', 'tripduration'])
df_2020 = df_2020.drop(columns=['start_lat', 'start_lng', 'end_lat', 'end_lng', 'rideable_type'])

### 4.2 Renomeando as colunas equivalentes

In [ ]:
df_2019 = df_2019.rename(columns={
    'trip_id': 'ride_id',
    'start_time': 'started_at',
    'end_time': 'ended_at',
    'from_station_id': 'start_station_id',
    'from_station_name': 'start_station_name',
    'to_station_id': 'end_station_id',
    'to_station_name': 'end_station_name',
    'usertype': 'member_casual'
})

### 4.3 Alinhando tipos de dados

In [ ]:
df_2019['ride_id'] = df_2019['ride_id'].astype(str)
df_2020['start_station_id'] = df_2020['start_station_id'].astype(str)
df_2020['end_station_id'] = df_2020['end_station_id'].astype(str)

### 4.4 Combinando os datasets

In [ ]:
df_completo = pd.concat([df_2019, df_2020], ignore_index=True)
df_completo.info()

### 4.5 Convertendo colunas de tempo para DateTime

In [ ]:
df_completo['started_at'] = pd.to_datetime(df_completo['started_at'])
df_completo['ended_at'] = pd.to_datetime(df_completo['ended_at'])

### 4.6 Feature Engineering: duração da viagem e dia da semana

In [ ]:
df_completo['ride_length'] = (df_completo['ended_at'] - df_completo['started_at']).dt.total_seconds()

### 4.7 Padronizando categorias de usuário

O nome que representava o tipo de usuário passou por mudanças no banco de dados entre 2019 e 2020:  
`Subscriber` → `member` e `Customer` → `casual`.

In [ ]:
df_completo['member_casual'] = df_completo['member_casual'].replace({
    'Subscriber': 'member',
    'Customer': 'casual'
})

### 4.8 Removendo anomalias

Existiam viagens com `ride_length` negativo ou zero, fisicamente impossíveis — geradas por erros de sistema ou testes de manutenção. Essas entradas são removidas para garantir a integridade das médias.

In [ ]:
df_completo['ride_length'].describe()

In [ ]:
df_completo = df_completo[df_completo['ride_length'] > 0]

In [ ]:
df_completo['day_of_week'] = df_completo['started_at'].dt.day_name()

---
## 5. Análise Exploratória

### 5.1 Distribuição por tipo de usuário

In [ ]:
df_completo['member_casual'].value_counts()

### 5.2 Duração média de viagem por tipo de usuário

In [ ]:
media_seg = df_completo.groupby('member_casual')['ride_length'].mean()
media_min = media_seg / 60

print("Duração média por tipo de usuário:")
for tipo, minutos in media_min.items():
    print(f"  {tipo}: {minutos:.1f} min ({media_seg[tipo]:.0f} seg)")

### 5.3 Volume de viagens por dia da semana e tipo de usuário

In [ ]:
ordem_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_completo['day_of_week'] = pd.Categorical(df_completo['day_of_week'], categories=ordem_dias, ordered=True)

viagens_por_dia = (
    df_completo
    .groupby(['member_casual', 'day_of_week'], observed=False)['ride_id']
    .count()
    .reset_index()
)
display(viagens_por_dia.sort_values(['member_casual', 'day_of_week']))

---
## 6. Visualizações

### 6.1 Volume de viagens por dia da semana

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

cores = {'member': '#2196F3', 'casual': '#FF9800'}
dias_pt = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']

for ax, tipo in zip(axes, ['member', 'casual']):
    dados = viagens_por_dia[viagens_por_dia['member_casual'] == tipo]
    ax.bar(dias_pt, dados['ride_id'], color=cores[tipo], edgecolor='white', linewidth=0.8)
    ax.set_title(f'Membro Anual' if tipo == 'member' else 'Usuário Casual', fontsize=13, fontweight='bold')
    ax.set_xlabel('Dia da Semana')
    ax.set_ylabel('Nº de Viagens')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'.replace(',', '.')))
    ax.tick_params(axis='x', rotation=0)

fig.suptitle('Volume de Viagens por Dia da Semana', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6.2 Duração média de viagem: Member vs. Casual

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

tipos = ['Usuário Casual', 'Membro Anual']
valores = [media_min['casual'], media_min['member']]
bar_cores = ['#FF9800', '#2196F3']

bars = ax.barh(tipos, valores, color=bar_cores, edgecolor='white', height=0.5)

for bar, val in zip(bars, valores):
    ax.text(val + 1, bar.get_y() + bar.get_height() / 2,
            f'{val:.0f} min', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Duração Média (minutos)')
ax.set_title('Duração Média de Viagem por Tipo de Usuário', fontsize=13, fontweight='bold')
ax.set_xlim(0, max(valores) * 1.2)
sns.despine(left=True, bottom=False)
plt.tight_layout()
plt.show()

---
## 7. Exportando Dados Limpos

In [ ]:
df_completo.to_csv('cyclistic_dados_limpos.csv', index=False)
print(f"Arquivo exportado com {len(df_completo):,} registros.")

---
## 8. Conclusão

A análise dos dados do Q1 2019 e 2020 revelou **duas personas de uso completamente distintas**:

| | Membro Anual | Usuário Casual |
|---|---|---|
| **Volume** | 91% das viagens | 9% das viagens |
| **Duração média** | ~13 min | ~85 min |
| **Padrão semanal** | Pico seg–sex | Pico sáb–dom |
| **Perfil** | Deslocamento diário | Lazer e passeios |

### Recomendações Estratégicas

Como o usuário casual não usa o serviço para rotina diária, um plano anual tradicional não atende às suas necessidades. Recomenda-se:

1. **Plano Anual de Fim de Semana** — produto desenhado especificamente para sábados e domingos.
2. **Marketing Geofocalizado** — anúncios e promotores nas estações próximas a orlas e parques, exclusivamente nos finais de semana.
3. **Gamificação de Retenção** — recompensas no app para viagens acima de 60 minutos exclusivas a membros, criando incentivo imediato para conversão.